![Logo](https://raw.githubusercontent.com/BartaZoltan/deep-reinforcement-learning-course/main/notebooks/shared_assets/logo.png)

# Practice 3 Homework: Monte Carlo on Blackjack

**Developers:** Domonkos Nagy, Balazs Nagy, Zoltan Barta  
**Date:** 2026-03-04  
**Version:** 2025-26/2

[<img src="https://colab.research.google.com/assets/colab-badge.svg">](https://colab.research.google.com/github/BartaZoltan/deep-reinforcement-learning-course/blob/main/notebooks/sessions/3_monte_carlo_methods/monte_carlo_methods_homework.ipynb)

## Summary

This homework extends the Monte Carlo material from the practice notebook to the classic **Blackjack** example from Sutton & Barto (Chapter 5).

Content outline:
- episodic interaction in `Blackjack-v1`,
- first-visit Monte Carlo prediction for a fixed policy,
- epsilon-greedy Monte Carlo control over state-action values,
- policy and value visualization for usable / non-usable ace states.


## Task Description: Blackjack with Monte Carlo Methods

In this homework, you will implement Monte Carlo prediction and control for the **Blackjack** environment.

The state is represented by:
- `player_sum` (current sum of the player's hand),
- `dealer_card` (dealer's visible card),
- `usable_ace` (whether the player holds a usable ace).

The action space is:
- `0`: stick,
- `1`: hit.

Your goal is to:
1. generate full episodes under a fixed policy,
2. estimate action values with **first-visit Monte Carlo prediction**,
3. extend this to **Monte Carlo control** with an epsilon-greedy policy,
4. inspect the learned value function and the greedy policy.

### Target learning setup

Use the same Monte Carlo logic as in the session notebook:
- wait until the episode ends,
- move backward through the trajectory,
- compute returns recursively,
- and average returns for visited state-action pairs.

For prediction, start from a simple baseline policy such as:
- **stick** if `player_sum >= 20`,
- **hit** otherwise.

For control, improve the policy over time using an **epsilon-greedy** action-selection rule.


In [ ]:
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import gymnasium as gym
except Exception as exc:
    raise ImportError("This homework requires gymnasium. Please install it before running the notebook.") from exc

sns.set_theme(style="whitegrid")
np.random.seed(42)


## Environment Setup

The next cells define the Blackjack environment and visualization helpers. You should not need to modify them.


In [ ]:
# DO NOT MODIFY THIS CELL

ACTION_NAMES = {0: "stick", 1: "hit"}

def make_env():
    return gym.make("Blackjack-v1")

def default_blackjack_policy(state):
    player_sum, dealer_card, usable_ace = state
    return 0 if player_sum >= 20 else 1

def state_key(state):
    player_sum, dealer_card, usable_ace = state
    return (int(player_sum), int(dealer_card), bool(usable_ace))


In [ ]:
# DO NOT MODIFY THIS CELL

def epsilon_greedy_probs(Q, state, epsilon=0.1):
    probs = np.ones(2, dtype=float) * (epsilon / 2.0)
    best_action = int(np.argmax(Q[state]))
    probs[best_action] += 1.0 - epsilon
    return probs

def greedy_action(Q, state):
    return int(np.argmax(Q[state]))

def sample_from_probs(probs):
    return int(np.random.choice(len(probs), p=probs))


In [ ]:
# DO NOT MODIFY THIS CELL

def value_grid_from_Q(Q, usable_ace=False):
    player_sums = range(12, 22)
    dealer_cards = range(1, 11)
    grid = np.zeros((len(player_sums), len(dealer_cards)), dtype=float)
    for i, player_sum in enumerate(player_sums):
        for j, dealer_card in enumerate(dealer_cards):
            state = (player_sum, dealer_card, usable_ace)
            grid[i, j] = np.max(Q[state])
    return grid

def policy_grid_from_Q(Q, usable_ace=False):
    player_sums = range(12, 22)
    dealer_cards = range(1, 11)
    grid = np.zeros((len(player_sums), len(dealer_cards)), dtype=int)
    for i, player_sum in enumerate(player_sums):
        for j, dealer_card in enumerate(dealer_cards):
            state = (player_sum, dealer_card, usable_ace)
            grid[i, j] = int(np.argmax(Q[state]))
    return grid

def plot_blackjack_values(Q, usable_ace=False, title="Blackjack value surface"):
    grid = value_grid_from_Q(Q, usable_ace=usable_ace)
    plt.figure(figsize=(8, 5))
    sns.heatmap(
        grid,
        cmap="viridis",
        xticklabels=list(range(1, 11)),
        yticklabels=list(range(12, 22)),
        cbar_kws={"label": "max_a Q(s,a)"},
    )
    plt.xlabel("Dealer showing")
    plt.ylabel("Player sum")
    plt.title(f"{title} | usable_ace={usable_ace}")
    plt.tight_layout()
    plt.show()

def plot_blackjack_policy(Q, usable_ace=False, title="Blackjack greedy policy"):
    grid = policy_grid_from_Q(Q, usable_ace=usable_ace)
    plt.figure(figsize=(8, 5))
    sns.heatmap(
        grid,
        cmap="coolwarm",
        xticklabels=list(range(1, 11)),
        yticklabels=list(range(12, 22)),
        cbar_kws={"label": "0=stick, 1=hit"},
        vmin=0,
        vmax=1,
    )
    plt.xlabel("Dealer showing")
    plt.ylabel("Player sum")
    plt.title(f"{title} | usable_ace={usable_ace}")
    plt.tight_layout()
    plt.show()


## Task 1: Generate a full Blackjack episode

Implement an episode generator that follows a given policy until the environment terminates.

Requirements:
- reset the environment,
- repeatedly choose an action from the current state,
- store `(state, action, reward)` tuples,
- stop when the episode terminates or truncates.


In [ ]:
def generate_episode(env, policy_fn):
    ########################################################################
    # TODO: roll out one full Blackjack episode under the provided policy.
    # Return a list of (state, action, reward) tuples.

    raise NotImplementedError("Task 1: implement generate_episode")
    ########################################################################


## Task 2: First-Visit Monte Carlo prediction

Estimate the action-value function of the fixed baseline Blackjack policy.

Requirements:
- use **first-visit** updates for state-action pairs,
- move backward through the episode,
- maintain running sums and counts of returns,
- return a dictionary-like `Q` mapping states to action values.


In [ ]:
def mc_first_visit_prediction(policy_fn, n_episodes=200_000, gamma=1.0):
    ########################################################################
    # TODO: estimate Q(s,a) for the fixed policy using first-visit MC.
    # Suggested structure:
    # - Q = defaultdict(lambda: np.zeros(2, dtype=float))
    # - returns_sum / returns_count with the same key structure
    # - reuse generate_episode(...)

    raise NotImplementedError("Task 2: implement mc_first_visit_prediction")
    ########################################################################


## Task 3: Monte Carlo control with epsilon-greedy improvement

Extend prediction into a full control loop.

Requirements:
- keep a tabular `Q(s,a)` estimate,
- generate episodes using the current epsilon-greedy policy,
- update visited state-action pairs with **first-visit** MC,
- improve the policy after each episode by acting epsilon-greedily with respect to `Q`.


In [ ]:
def mc_blackjack_control(n_episodes=500_000, epsilon=0.1, gamma=1.0):
    ########################################################################
    # TODO: implement epsilon-greedy on-policy MC control for Blackjack.
    # Return the learned Q table.

    raise NotImplementedError("Task 3: implement mc_blackjack_control")
    ########################################################################


## Task 4: Visualize the learned policy

Train the Monte Carlo control agent, then plot:
- value heatmaps for `usable_ace=False` and `usable_ace=True`,
- greedy policy heatmaps for the same two cases.

These plots should qualitatively resemble the Blackjack policy/value patterns discussed in the book.


In [ ]:
Q_blackjack = mc_blackjack_control(n_episodes=300_000, epsilon=0.1, gamma=1.0)

plot_blackjack_values(Q_blackjack, usable_ace=False, title="Estimated value")
plot_blackjack_values(Q_blackjack, usable_ace=True, title="Estimated value")

plot_blackjack_policy(Q_blackjack, usable_ace=False, title="Greedy policy")
plot_blackjack_policy(Q_blackjack, usable_ace=True, title="Greedy policy")
